# K4TQ-G3 "Ginna 3": as fast as a 15-inch FPV quad with its payload

Ginna's airframe (Freddy × 1.35) with the Jetson Orin NX and the perception, take-off mass ≤ 12.5 kg, the payload re-sized to the required 2.1 kg (× 1.32 to stay ≤ 12.5 kg); the drive swept for the top speed — **T-Motor MN605-S KV170 + Aeronaut CAM Carbon 17x13 folding, on 12S3P**.

**The target**: the 15-inch FPV quad's top speed carrying a payload (`k4tq_quads.fpv15`, the same catalogue and drive
model: 15x8 on MN5006 KV450, 6S2P Li-ion) — ≈34.5 m/s (124 km/h); K4TQ-G3 must match or exceed it with four motors
running, with a 5 % margin (the model's 1 m/s grid and its uncertainty).

**Off the shelf**: only parts sold today — T-Motor Navigator motors within their rated voltage (MN505-S is 6S only;
MN601-S / MN605-S / MN801-S 6–12S), Aeronaut CAM Carbon folding blades (16x13, 17x11, 17x13, 18x13, 18x9 on the 42 mm
hub) and T-Motor's folding VTOL props (MF2009 20.4x7.3, FA22.2x7.2), packs of Molicel P45B cells.

**The rules** (`k4tq_fast.sweep` / `pick`): a motor bigger than Ginna autonomous (G2)'s; the still-air range at least 90 % of
Ginna autonomous (G2)'s ("more or less the same"; a bigger pack allowed); T/W ≥ 1.6 on the sagged pack, most loaded motor
≤ 75 % in the hover; R3 closes (10 km radius, 2 min hover, 20 % reserve, 8 m/s wind) with ≥ 20 Wh to spare; the root's
actuator (sized for the top speed + 10 %), bearings and locking pin within their ratings; the most payload that passes within 12.5 kg and Freddy's stall speed (≤ 18 m/s).

Why the drive and not the airframe: Ginna autonomous's top speed is its propeller's pitch speed, not its power —
the drag at 35 m/s is only ≈0.5 kW of thrust power. A higher pitch (or a higher pack voltage) moves it.
Labels: ESTIMATE / CALC (MOCK BEMT under `DEBUG = True`); the new parts are "to verify".

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import pandas as pd
from IPython.display import Image, display
import k4tq, k4tq_sizing as sz, k4tq_fast as kf, k4tq_quads as kq
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 50)
LETTER, PARENT = "G3", "G2"
print(k4tq.params(LETTER)["title"])
print("target: the 15-inch FPV quad's top speed with payload", kf.target_speed(), "m/s")

## 1. The sweep (motor × propeller × pack) — from `data/k4tq/sweep_G3.csv`; re-run below (10–30 min)

In [ ]:
sw = pd.read_csv(sz.DATA / "sweep_G3.csv")
sw.attrs["target speed [m/s]"] = kf.target_speed()
# sw = kf.sweep(LETTER); sw.to_csv(sz.DATA / "sweep_G3.csv", index=False)       # the full sweep again
cols = ["motor", "prop", "pack", "top speed [m/s]", "take-off mass [kg]", "payload [kg]", "T/W sagged", "R3 margin [Wh]", "still-air range [km]",
        "hover [W]", "stall [m/s]", "actuator reserve", "meets the target", "note"]
sw[[c for c in cols if c in sw]].sort_values("top speed [m/s]", ascending=False).round(2)

In [ ]:
print("the pick:"); kf.pick(LETTER, sw)

## 2. Against Ginna autonomous (G2) and the 15-inch FPV quad

In [ ]:
df = kf.comparison(designs=(PARENT, LETTER))
df.T

## 3. Power and range against speed

In [ ]:
Image(str(kf.figure(designs=(PARENT, LETTER))))

## 4. The parts: what to buy

In [ ]:
p = k4tq.params(LETTER)
cat = k4tq.CATALOGUE
pd.DataFrame({"motor ×4": cat["motor"][p["motor"]], "propeller ×4 (+2 spares)": cat["propeller"][p["prop"]], "ESC ×4": cat[p["esc"]],
              "pack": cat["battery"][p["battery"]]}).T[["mass_kg", "price_eur", "source"]]

## 5. The root and the hover

In [ ]:
v = sz.vehicle(LETTER)
r = sz.root_loads(v)
display(pd.Series({k: r[k] for k in ("case governing the bending", "root moment ultimate [N·m]", "actuator design torque [N·m]", "actuator available at the spar [N·m]",
                    "actuator reserve", "bearing static reserve", "locking pin shear stress, double shear [MPa]", "spar chosen")}))
pd.Series(sz.hover(v))

## Reading it

- **Is it possible?** Yes. ≈40 m/s (144 km/h) against the FPV quad's 121 km/h with 2 kg — 19 % faster — with the
  Jetson and the required 2.1 kg payload (G2: 2.3 kg; the airframe × 1.32 instead of 1.35 keeps 12.47 kg, the stall at 18 m/s); **the range is not lost, it grows**: 82 km
  still-air against G2's 57 km (R3 radius 17.8 against 13.7 km). A smaller pack would not buy the payload back — on the
  12S2P the R3 margin collapses and the payload falls to 1.7 kg — so the extra range comes free.
- **What it took** (all on the shelf): the MN605-S KV170 (350 g, 35 A, 6–12S, +0.1 kg per motor) and the Aeronaut CAM
  Carbon 17x13 folding blades (a smaller disc than Ginna's 20", the higher pitch) on the 12S3P. Hover 2.55 kW (G2: 2.0).
- **Watch**: the tilt actuator at 44 m/s (top + 10 %) has a reserve of ≈1.07 — the 6.5 N·m servo class (`root_M`, ≈+50 g
  each) is the first Stage 2 change; G2's own drive (the U8 II KV170 + 20x10) is an assumed class, not a shelf item — its
  real twin is the MN601-S KV170; divergence / flutter at 40 m/s (Stage 4).
- **The FPV still wins** the hover and the mass (4.3 kg against 12.5 kg).